# Groundwater Prediction Pipeline (Pune)

This notebook upgrades the groundwater forecasting workflow with richer lag engineering, robust time-series validation, and a model leaderboard that compares baseline and ML challengers.

Upgrades in this version:
1. Expanded lag and rolling-window feature engineering
2. Station-aware residual modeling (baseline plus correction)
3. Temporal split plus rolling-origin backtest
4. Clear metric tables (MAE, RMSE, R2, and improvement vs baseline)
5. Champion-model driven recursive forecasting to Gold Delta

In [ ]:
import os
import math
from datetime import timedelta

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.regression import GBTRegressor, LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

try:
    from delta import configure_spark_with_delta_pip
except ImportError:
    configure_spark_with_delta_pip = None

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IS_DATABRICKS:
    spark = globals().get("spark")
    if spark is None:
        spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()

    NOTEBOOK_CWD = os.getcwd()
    PROJECT_ROOT = "/Workspace"
    INPUT_PATH = "dbfs:/Volumes/iitb/default/bhujal_mitra_vol/Maharashtra_Pune_Filtered.csv"
    GOLD_PATH = "dbfs:/Volumes/iitb/default/bhujal_mitra_vol/gold/groundwater_prediction_gold"
    GOLD_TABLE = None
else:
    if configure_spark_with_delta_pip is None:
        raise ImportError("delta-spark is required for local execution.")

    builder = (
        SparkSession.builder
        .appName("GroundWaterPrediction-Pipeline")
        .master("local[*]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.shuffle.partitions", "64")
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

    NOTEBOOK_CWD = os.getcwd()
    if os.path.exists(os.path.join(NOTEBOOK_CWD, "data")):
        PROJECT_ROOT = NOTEBOOK_CWD
    else:
        PROJECT_ROOT = os.path.dirname(NOTEBOOK_CWD)

    INPUT_PATH = os.path.join(PROJECT_ROOT, "data", "Maharashtra_Pune_Filtered.csv")
    GOLD_PATH = os.path.join(PROJECT_ROOT, "data", "gold", "groundwater_prediction_gold")
    GOLD_TABLE = "bhujal_mitra.groundwater_prediction_gold"

FORECAST_HORIZON_DAYS = 30

LAG_STEPS = [1, 2, 3, 7, 14, 30]
RAIN_LAG_STEPS = [1, 3, 7, 14, 30]
ROLL_WINDOWS = [7, 14, 30]
CV_QUANTILES = [0.60, 0.70, 0.80]

print("Running on Databricks:", IS_DATABRICKS)
print("Notebook cwd:", NOTEBOOK_CWD)
print("Resolved project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("Gold path:", GOLD_PATH)
if GOLD_TABLE is not None:
    print("Gold table:", GOLD_TABLE)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/29 01:50:53 WARN Utils: Your hostname, mridulUbuntu, resolves to a loopback address: 127.0.1.1; using 10.51.26.78 instead (on interface wlp2s0)
26/03/29 01:50:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/mridul-sharma/Desktop/bhujal-mitra/.venv-1/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/mridul-sharma/.ivy2.5.2/cache
The jars for the packages stored in: /home/mridul-sharma/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-09fce844-29aa-4c47-b4ee-ff8f23d60911;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf

Notebook cwd: /home/mridul-sharma/Desktop/bhujal-mitra/src
Resolved project root: /home/mridul-sharma/Desktop/bhujal-mitra
Input path: /home/mridul-sharma/Desktop/bhujal-mitra/data/Maharashtra_Pune_Filtered.csv
Gold path: /home/mridul-sharma/Desktop/bhujal-mitra/data/gold/groundwater_prediction_gold


In [ ]:
read_input_path = INPUT_PATH

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(read_input_path)
)

df = (
    raw_df
    .withColumn("station_id", F.col("station_id").cast("string"))
    .withColumn("datetime", F.to_date("datetime"))
    .withColumn("groundwater_level_target", F.col("groundwater_level_target").cast("double"))
    .withColumn("rainfall", F.coalesce(F.col("rainfall").cast("double"), F.lit(0.0)))
    .filter(F.col("datetime").isNotNull() & F.col("groundwater_level_target").isNotNull())
)

print(f"Rows after load: {df.count()}")
print(f"Stations: {df.select('station_id').distinct().count()}")
print("Date range:")
df.select(F.min("datetime").alias("start"), F.max("datetime").alias("end")).show(truncate=False)

df.printSchema()
df.show(5, truncate=False)

Rows after load: 14641
Stations: 94
Date range:
+----------+----------+
|start     |end       |
+----------+----------+
|1994-01-05|2025-09-25|
+----------+----------+

root
 |-- station_id: string (nullable = true)
 |-- time_idx: integer (nullable = true)
 |-- datetime: date (nullable = true)
 |-- groundwater_level_target: double (nullable = true)
 |-- rainfall: double (nullable = false)
 |-- t2m_avg: double (nullable = true)
 |-- t2m_max: double (nullable = true)
 |-- t2m_min: double (nullable = true)
 |-- month: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- wellDepth: double (nullable = true)
 |-- wellAquiferType_encoded: integer (nullable = true)

+----------+--------+----------+------------------------+--------+-------+-------+-------+-----+--------+---------+---------+-----------------------+
|station_id|time_idx|datetime  |groundwater_level_target|rainfall|t2m_avg|t2m_max|t2m_min|month|latitude|longitude|wellDepth

In [3]:
null_exprs = [
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["station_id", "datetime", "groundwater_level_target", "rainfall"]
]
null_summary = df.select(*null_exprs)
print("Null summary:")
null_summary.show(truncate=False)

duplicate_station_date = df.groupBy("station_id", "datetime").count().filter(F.col("count") > 1)
dup_count = duplicate_station_date.count()
print(f"Duplicate station-date keys: {dup_count}")

coverage_df = (
    df.groupBy("station_id")
    .agg(
        F.count("*").alias("records"),
        F.min("datetime").alias("start_date"),
        F.max("datetime").alias("end_date"),
        F.stddev_pop("groundwater_level_target").alias("target_std")
    )
    .orderBy(F.desc("records"))
)

print("Top 10 stations by history length:")
coverage_df.show(10, truncate=False)

print("Top 10 stations by target volatility:")
coverage_df.orderBy(F.desc("target_std")).show(10, truncate=False)

print("Coverage summary:")
coverage_df.select(
    F.min("records").alias("min_records"),
    F.max("records").alias("max_records"),
    F.avg("records").alias("avg_records")
).show(truncate=False)

Null summary:
+----------+--------+------------------------+--------+
|station_id|datetime|groundwater_level_target|rainfall|
+----------+--------+------------------------+--------+
|0         |0       |0                       |0       |
+----------+--------+------------------------+--------+

Duplicate station-date keys: 0
Top 10 stations by history length:
+----------+-------+----------+----------+------------------+
|station_id|records|start_date|end_date  |target_std        |
+----------+-------+----------+----------+------------------+
|CGWBNG0549|409    |2023-04-19|2025-01-06|1.186260309889294 |
|CGWBNG0541|402    |2023-05-22|2025-09-06|4.175342031477917 |
|CGWBNG0596|317    |2023-06-17|2025-06-28|5.435014069792508 |
|CGWBNG0579|305    |2023-07-19|2025-09-14|19.281577566814125|
|CGWJPR0202|297    |2024-05-24|2025-08-25|2.7963660562858985|
|CGWBNG0583|295    |2023-05-04|2024-11-23|0.6685478980427656|
|CGWBNG0587|292    |2023-06-05|2025-06-03|0.747427818746448 |
|CGWJPR0246|286    

In [4]:
daily_df = (
    df.groupBy("station_id", "datetime")
    .agg(
        F.avg("groundwater_level_target").alias("groundwater_level_target"),
        F.avg("rainfall").alias("rainfall")
    )
    .orderBy("station_id", "datetime")
)

w_station = Window.partitionBy("station_id").orderBy("datetime")
feature_df = daily_df

for step in LAG_STEPS:
    feature_df = feature_df.withColumn(
        f"lag_{step}", F.lag("groundwater_level_target", step).over(w_station)
    )

for step in RAIN_LAG_STEPS:
    feature_df = feature_df.withColumn(
        f"rain_lag_{step}", F.lag("rainfall", step).over(w_station)
    )

for window_size in ROLL_WINDOWS:
    w_hist = w_station.rowsBetween(-window_size, -1)
    feature_df = feature_df.withColumn(
        f"roll_mean_{window_size}", F.avg("groundwater_level_target").over(w_hist)
    )
    feature_df = feature_df.withColumn(
        f"roll_std_{window_size}", F.stddev_pop("groundwater_level_target").over(w_hist)
    )
    feature_df = feature_df.withColumn(
        f"roll_rain_mean_{window_size}", F.avg("rainfall").over(w_hist)
    )
    feature_df = feature_df.withColumn(
        f"roll_rain_sum_{window_size}", F.sum("rainfall").over(w_hist)
    )

feature_df = (
    feature_df
    .withColumn("month", F.month("datetime"))
    .withColumn("day_of_year", F.dayofyear("datetime"))
    .withColumn("month_sin", F.sin(F.lit(2.0 * math.pi) * F.col("month") / F.lit(12.0)))
    .withColumn("month_cos", F.cos(F.lit(2.0 * math.pi) * F.col("month") / F.lit(12.0)))
    .withColumn("doy_sin", F.sin(F.lit(2.0 * math.pi) * F.col("day_of_year") / F.lit(365.0)))
    .withColumn("doy_cos", F.cos(F.lit(2.0 * math.pi) * F.col("day_of_year") / F.lit(365.0)))
    .withColumn("date_ord", F.unix_date("datetime"))
)

feature_cols = []
for step in LAG_STEPS:
    feature_cols.append(f"lag_{step}")
for step in RAIN_LAG_STEPS:
    feature_cols.append(f"rain_lag_{step}")
for window_size in ROLL_WINDOWS:
    feature_cols.extend([
        f"roll_mean_{window_size}",
        f"roll_std_{window_size}",
        f"roll_rain_mean_{window_size}",
        f"roll_rain_sum_{window_size}",
    ])
feature_cols.extend(["rainfall", "month_sin", "month_cos", "doy_sin", "doy_cos"])

model_df = (
    feature_df
    .dropna(subset=feature_cols + ["groundwater_level_target"])
    .withColumn("residual_target", F.col("groundwater_level_target") - F.col("lag_1"))
)

print(f"Rows available for modeling: {model_df.count()}")
model_df.select(F.min("datetime").alias("model_start"), F.max("datetime").alias("model_end")).show(truncate=False)
print("Feature count:", len(feature_cols))
model_df.select("station_id", "datetime", "groundwater_level_target", *feature_cols[:12]).show(5, truncate=False)

Rows available for modeling: 11821
+-----------+----------+
|model_start|model_end |
+-----------+----------+
|2000-08-25 |2025-09-25|
+-----------+----------+

Feature count: 28
+---------------+----------+------------------------+-----+-----+-----+-----+------+------+----------+----------+----------+-----------+-----------+------------------+
|station_id     |datetime  |groundwater_level_target|lag_1|lag_2|lag_3|lag_7|lag_14|lag_30|rain_lag_1|rain_lag_3|rain_lag_7|rain_lag_14|rain_lag_30|roll_mean_7       |
+---------------+----------+------------------------+-----+-----+-----+-----+------+------+----------+----------+----------+-----------+-----------+------------------+
|153145076380001|2001-01-05|3.84                    |2.97 |3.6  |2.3  |4.68 |7.52  |5.76  |0.3       |0.06      |3.36      |0.65       |0.03       |3.758571428571429 |
|153145076380001|2001-05-15|5.43                    |3.84 |2.97 |3.6  |6.46 |3.15  |5.24  |0.0       |2.58      |3.72      |0.0        |1.95       |3

In [ ]:
split_ord = model_df.approxQuantile("date_ord", [0.8], 0.0)[0]

train_df = model_df.filter(F.col("date_ord") <= split_ord)
valid_df = model_df.filter(F.col("date_ord") > split_ord)

cv_cutoffs = model_df.approxQuantile("date_ord", CV_QUANTILES, 0.0)
max_date_ord = model_df.select(F.max("date_ord").alias("max_date_ord")).collect()[0]["max_date_ord"]

print(f"Train rows: {train_df.count()}")
print(f"Validation rows: {valid_df.count()}")
print("Train period:")
train_df.select(F.min("datetime").alias("start"), F.max("datetime").alias("end")).show(truncate=False)
print("Validation period:")
valid_df.select(F.min("datetime").alias("start"), F.max("datetime").alias("end")).show(truncate=False)

print("Rolling backtest cutoffs (date_ord):", cv_cutoffs)

Train rows: 9472
Validation rows: 2349
Train period:
+----------+----------+
|start     |end       |
+----------+----------+
|2000-08-25|2024-09-11|
+----------+----------+

Validation period:
+----------+----------+
|start     |end       |
+----------+----------+
|2024-09-12|2025-09-25|
+----------+----------+

Rolling backtest cutoffs (date_ord): [19825.0, 19903.0, 19977.0]


In [6]:
def evaluate_regression(pred_df, prediction_col="prediction", label_col="groundwater_level_target"):
    rmse = RegressionEvaluator(
        labelCol=label_col, predictionCol=prediction_col, metricName="rmse"
    ).evaluate(pred_df)
    mae = RegressionEvaluator(
        labelCol=label_col, predictionCol=prediction_col, metricName="mae"
    ).evaluate(pred_df)
    r2 = RegressionEvaluator(
        labelCol=label_col, predictionCol=prediction_col, metricName="r2"
    ).evaluate(pred_df)
    return {"rmse": float(rmse), "mae": float(mae), "r2": float(r2)}


def build_residual_pipeline(regressor):
    station_indexer = StringIndexer(
        inputCol="station_id", outputCol="station_idx", handleInvalid="keep"
    )
    assembler = VectorAssembler(
        inputCols=["station_idx"] + feature_cols, outputCol="features_raw"
    )
    scaler = StandardScaler(
        inputCol="features_raw", outputCol="features", withMean=True, withStd=True
    )
    return Pipeline(stages=[station_indexer, assembler, scaler, regressor])


candidate_estimators = {
    "residual_gbt_v2": GBTRegressor(
        featuresCol="features",
        labelCol="residual_target",
        predictionCol="model_prediction",
        maxIter=90,
        maxDepth=3,
        stepSize=0.05,
        subsamplingRate=0.8,
        seed=42,
    ),
    "residual_linear_v2": LinearRegression(
        featuresCol="features",
        labelCol="residual_target",
        predictionCol="model_prediction",
        maxIter=200,
        regParam=0.001,
        elasticNetParam=0.0,
        standardization=False,
    ),
}

MODEL_REGISTRY = {}
metrics_rows = []

baseline_valid_pred = valid_df.withColumn("prediction", F.col("lag_1"))
baseline_metrics = evaluate_regression(baseline_valid_pred)
metrics_rows.append({
    "model_name": "naive_lag_1",
    **baseline_metrics,
})

for model_name, estimator in candidate_estimators.items():
    pipeline_model = build_residual_pipeline(estimator).fit(train_df)
    valid_pred = (
        pipeline_model.transform(valid_df)
        .withColumn("prediction", F.col("lag_1") + F.col("model_prediction"))
    )
    model_metrics = evaluate_regression(valid_pred)
    metrics_rows.append({
        "model_name": model_name,
        **model_metrics,
    })
    MODEL_REGISTRY[model_name] = pipeline_model

metrics_pd = pd.DataFrame(metrics_rows).sort_values(["mae", "rmse"]).reset_index(drop=True)
baseline_mae = float(metrics_pd.loc[metrics_pd["model_name"] == "naive_lag_1", "mae"].iloc[0])
baseline_rmse = float(metrics_pd.loc[metrics_pd["model_name"] == "naive_lag_1", "rmse"].iloc[0])

metrics_pd["mae_improvement_vs_baseline"] = baseline_mae - metrics_pd["mae"]
metrics_pd["rmse_improvement_vs_baseline"] = baseline_rmse - metrics_pd["rmse"]

CHAMPION_MODEL_NAME = str(metrics_pd.iloc[0]["model_name"])
print(f"Champion model on holdout: {CHAMPION_MODEL_NAME}")
print("Model leaderboard (lower MAE/RMSE is better):")
spark.createDataFrame(metrics_pd.round(6)).show(truncate=False)

if CHAMPION_MODEL_NAME == "naive_lag_1":
    champion_valid_pred = baseline_valid_pred
else:
    champion_valid_pred = (
        MODEL_REGISTRY[CHAMPION_MODEL_NAME]
        .transform(valid_df)
        .withColumn("prediction", F.col("lag_1") + F.col("model_prediction"))
    )

Champion model on holdout: naive_lag_1
Model leaderboard (lower MAE/RMSE is better):


+------------------+--------+--------+--------+---------------------------+----------------------------+
|model_name        |rmse    |mae     |r2      |mae_improvement_vs_baseline|rmse_improvement_vs_baseline|
+------------------+--------+--------+--------+---------------------------+----------------------------+
|naive_lag_1       |4.581371|0.390525|0.946085|0.0                        |0.0                         |
|residual_gbt_v2   |4.615982|0.458446|0.945267|-0.067921                  |-0.034611                   |
|residual_linear_v2|3.874897|1.143418|0.961431|-0.752893                  |0.706474                    |
+------------------+--------+--------+--------+---------------------------+----------------------------+



In [7]:
rolling_rows = []

for idx, cutoff in enumerate(cv_cutoffs):
    fold_end = cv_cutoffs[idx + 1] if idx + 1 < len(cv_cutoffs) else max_date_ord

    fold_train = model_df.filter(F.col("date_ord") <= cutoff)
    fold_valid = model_df.filter(
        (F.col("date_ord") > cutoff) & (F.col("date_ord") <= fold_end)
    )

    fold_valid_rows = fold_valid.count()
    if fold_valid_rows == 0:
        continue

    fold_baseline_pred = fold_valid.withColumn("prediction", F.col("lag_1"))
    fold_baseline_metrics = evaluate_regression(fold_baseline_pred)
    rolling_rows.append({
        "fold": idx + 1,
        "model_name": "naive_lag_1",
        "valid_rows": fold_valid_rows,
        **fold_baseline_metrics,
    })

    if CHAMPION_MODEL_NAME != "naive_lag_1":
        fold_model = build_residual_pipeline(
            candidate_estimators[CHAMPION_MODEL_NAME]
        ).fit(fold_train)

        fold_champion_pred = (
            fold_model.transform(fold_valid)
            .withColumn("prediction", F.col("lag_1") + F.col("model_prediction"))
        )
        fold_champion_metrics = evaluate_regression(fold_champion_pred)
        rolling_rows.append({
            "fold": idx + 1,
            "model_name": CHAMPION_MODEL_NAME,
            "valid_rows": fold_valid_rows,
            **fold_champion_metrics,
        })

rolling_pd = pd.DataFrame(rolling_rows)

if rolling_pd.empty:
    print("No rolling backtest rows produced.")
    rolling_summary_pd = pd.DataFrame()
else:
    print("Rolling backtest metrics by fold:")
    spark.createDataFrame(rolling_pd.round(6)).show(truncate=False)

    rolling_summary_pd = (
        rolling_pd.groupby("model_name")[["mae", "rmse", "r2"]]
        .mean()
        .reset_index()
        .sort_values(["mae", "rmse"]) 
        .reset_index(drop=True)
    )

    if "naive_lag_1" in rolling_summary_pd["model_name"].values:
        rolling_baseline_mae = float(
            rolling_summary_pd.loc[
                rolling_summary_pd["model_name"] == "naive_lag_1", "mae"
            ].iloc[0]
        )
        rolling_baseline_rmse = float(
            rolling_summary_pd.loc[
                rolling_summary_pd["model_name"] == "naive_lag_1", "rmse"
            ].iloc[0]
        )
        rolling_summary_pd["mae_improvement_vs_baseline"] = (
            rolling_baseline_mae - rolling_summary_pd["mae"]
        )
        rolling_summary_pd["rmse_improvement_vs_baseline"] = (
            rolling_baseline_rmse - rolling_summary_pd["rmse"]
        )

    print("Rolling backtest summary (mean across folds):")
    spark.createDataFrame(rolling_summary_pd.round(6)).show(truncate=False)

Rolling backtest metrics by fold:
+----+-----------+----------+--------+--------+--------+
|fold|model_name |valid_rows|rmse    |mae     |r2      |
+----+-----------+----------+--------+--------+--------+
|1   |naive_lag_1|1183      |1.837999|0.274172|0.997617|
|2   |naive_lag_1|1188      |1.103622|0.280998|0.997885|
|3   |naive_lag_1|2349      |4.581371|0.390525|0.946085|
+----+-----------+----------+--------+--------+--------+

Rolling backtest summary (mean across folds):
+-----------+--------+--------+--------+---------------------------+----------------------------+
|model_name |mae     |rmse    |r2      |mae_improvement_vs_baseline|rmse_improvement_vs_baseline|
+-----------+--------+--------+--------+---------------------------+----------------------------+
|naive_lag_1|0.315232|2.507664|0.980529|0.0                        |0.0                         |
+-----------+--------+--------+--------+---------------------------+----------------------------+



In [8]:
residual_df = (
    champion_valid_pred
    .withColumn("error", F.col("prediction") - F.col("groundwater_level_target"))
    .withColumn("abs_error", F.abs(F.col("error")))
    .withColumn("sq_error", F.pow(F.col("error"), 2))
)

print("Champion residual summary:")
residual_df.select(
    F.avg("error").alias("mean_error"),
    F.avg("abs_error").alias("mean_abs_error"),
    F.sqrt(F.avg("sq_error")).alias("root_mean_sq_error")
).show(truncate=False)

print("Top 10 stations by holdout MAE:")
residual_df.groupBy("station_id").agg(
    F.avg("abs_error").alias("station_mae"),
    F.count("*").alias("rows")
).orderBy(F.desc("station_mae")).show(10, truncate=False)

print("Worst 15 holdout errors (for debugging):")
residual_df.select(
    "station_id",
    "datetime",
    "groundwater_level_target",
    "prediction",
    "error",
    "abs_error"
).orderBy(F.desc("abs_error")).show(15, truncate=False)

if CHAMPION_MODEL_NAME == "naive_lag_1":
    final_model = None
    print("Final forecasting model: naive_lag_1 (no additional model fit required).")
else:
    final_model = build_residual_pipeline(
        candidate_estimators[CHAMPION_MODEL_NAME]
    ).fit(model_df)
    print(f"Final forecasting model trained: {CHAMPION_MODEL_NAME}")

station_monthly_rain = (
    daily_df
    .withColumn("month", F.month("datetime"))
    .groupBy("station_id", "month")
    .agg(F.avg("rainfall").alias("avg_monthly_rainfall"))
)

global_monthly_rain = (
    daily_df
    .withColumn("month", F.month("datetime"))
    .groupBy("month")
    .agg(F.avg("rainfall").alias("global_avg_monthly_rainfall"))
)

station_monthly_pd = station_monthly_rain.toPandas()
global_monthly_pd = global_monthly_rain.toPandas()

station_monthly_lookup = {
    (row.station_id, int(row.month)): float(row.avg_monthly_rainfall)
    for row in station_monthly_pd.itertuples(index=False)
}
global_monthly_lookup = {
    int(row.month): float(row.global_avg_monthly_rainfall)
    for row in global_monthly_pd.itertuples(index=False)
}

history_pd = (
    daily_df
    .select("station_id", "datetime", "groundwater_level_target", "rainfall")
    .orderBy("station_id", "datetime")
    .toPandas()
)
history_pd["datetime"] = pd.to_datetime(history_pd["datetime"])

station_histories = {}
for station_id, group in history_pd.groupby("station_id"):
    group = group.sort_values("datetime")
    station_histories[station_id] = [
        {
            "datetime": row.datetime.date(),
            "groundwater_level_target": float(row.groundwater_level_target),
            "rainfall": float(row.rainfall),
        }
        for row in group.itertuples(index=False)
    ]

print(f"Stations prepared for forecasting: {len(station_histories)}")

Champion residual summary:
+--------------------+-------------------+------------------+
|mean_error          |mean_abs_error     |root_mean_sq_error|
+--------------------+-------------------+------------------+
|0.011508443309209594|0.39052504611891586|4.581370806835128 |
+--------------------+-------------------+------------------+

Top 10 stations by holdout MAE:
+----------+-------------------+----+
|station_id|station_mae        |rows|
+----------+-------------------+----+
|CGWJPR0266|9.61408333333333   |40  |
|CGWBNG0579|1.2439179755671903 |191 |
|CGWBNG0555|0.7591526104417672 |83  |
|CGWBNG0498|0.3279062500000001 |80  |
|CGWBNG0541|0.2823094017094018 |195 |
|CGWBNG0596|0.2353787878787879 |143 |
|CGWNAG2007|0.17575            |160 |
|CGWJPR0202|0.12957219251336896|187 |
|CGWNAG2202|0.11726495726495728|78  |
|CGWJPR0246|0.08272093023255819|215 |
+----------+-------------------+----+
only showing top 10 rows
Worst 15 holdout errors (for debugging):
+----------+----------+---------

In [9]:
def safe_lag(series, step):
    if len(series) >= step:
        return float(series[-step])
    return float(series[-1])


def safe_roll_mean(series, window_size):
    segment = series[-window_size:] if len(series) >= window_size else series
    return float(np.mean(segment))


def safe_roll_std(series, window_size):
    segment = series[-window_size:] if len(series) >= window_size else series
    if len(segment) <= 1:
        return 0.0
    return float(np.std(segment))


future_records = []

for step in range(1, FORECAST_HORIZON_DAYS + 1):
    step_rows = []

    for station_id, history in station_histories.items():
        next_date = history[-1]["datetime"] + timedelta(days=1)
        month_value = next_date.month
        day_of_year = next_date.timetuple().tm_yday

        assumed_rainfall = station_monthly_lookup.get(
            (station_id, month_value),
            global_monthly_lookup.get(month_value, 0.0)
        )

        target_series = [h["groundwater_level_target"] for h in history]
        rain_series = [h["rainfall"] for h in history]

        row = {
            "station_id": station_id,
            "forecast_date": next_date.isoformat(),
            "step_ahead_days": step,
            "assumed_rainfall": float(assumed_rainfall),
            "rainfall": float(assumed_rainfall),
            "month_sin": float(math.sin((2.0 * math.pi * month_value) / 12.0)),
            "month_cos": float(math.cos((2.0 * math.pi * month_value) / 12.0)),
            "doy_sin": float(math.sin((2.0 * math.pi * day_of_year) / 365.0)),
            "doy_cos": float(math.cos((2.0 * math.pi * day_of_year) / 365.0)),
            "date_ord": int((pd.Timestamp(next_date) - pd.Timestamp("1970-01-01")) // pd.Timedelta("1D")),
        }

        for lag_step in LAG_STEPS:
            row[f"lag_{lag_step}"] = safe_lag(target_series, lag_step)
        for lag_step in RAIN_LAG_STEPS:
            row[f"rain_lag_{lag_step}"] = safe_lag(rain_series, lag_step)
        for window_size in ROLL_WINDOWS:
            row[f"roll_mean_{window_size}"] = safe_roll_mean(target_series, window_size)
            row[f"roll_std_{window_size}"] = safe_roll_std(target_series, window_size)
            row[f"roll_rain_mean_{window_size}"] = safe_roll_mean(rain_series, window_size)
            row[f"roll_rain_sum_{window_size}"] = float(
                np.sum(rain_series[-window_size:]) if len(rain_series) >= window_size else np.sum(rain_series)
            )

        if CHAMPION_MODEL_NAME == "naive_lag_1":
            predicted_level = row["lag_1"]
            station_histories[station_id].append({
                "datetime": next_date,
                "groundwater_level_target": predicted_level,
                "rainfall": float(assumed_rainfall),
            })
            future_records.append({
                "station_id": station_id,
                "forecast_date": next_date.isoformat(),
                "step_ahead_days": step,
                "assumed_rainfall": float(assumed_rainfall),
                "predicted_groundwater_level": float(predicted_level),
            })
        else:
            step_rows.append(row)

    if CHAMPION_MODEL_NAME != "naive_lag_1":
        step_df = spark.createDataFrame(pd.DataFrame(step_rows))
        step_predictions = (
            final_model.transform(step_df)
            .withColumn("forecast_date", F.to_date("forecast_date"))
            .withColumn("predicted_groundwater_level", F.col("lag_1") + F.col("model_prediction"))
            .select(
                "station_id",
                "forecast_date",
                "step_ahead_days",
                "assumed_rainfall",
                "predicted_groundwater_level"
            )
        )

        step_pd = step_predictions.toPandas()
        for pred_row in step_pd.itertuples(index=False):
            station_id = pred_row.station_id
            forecast_date = pd.to_datetime(pred_row.forecast_date).date()
            predicted_level = float(pred_row.predicted_groundwater_level)
            assumed_rainfall = float(pred_row.assumed_rainfall)

            station_histories[station_id].append({
                "datetime": forecast_date,
                "groundwater_level_target": predicted_level,
                "rainfall": assumed_rainfall,
            })
            future_records.append({
                "station_id": station_id,
                "forecast_date": forecast_date.isoformat(),
                "step_ahead_days": int(pred_row.step_ahead_days),
                "assumed_rainfall": assumed_rainfall,
                "predicted_groundwater_level": predicted_level,
            })

future_predictions_df = (
    spark.createDataFrame(pd.DataFrame(future_records))
    .withColumn("forecast_date", F.to_date("forecast_date"))
)

print(f"Future forecast rows generated: {future_predictions_df.count()}")
future_predictions_df.show(10, truncate=False)

Future forecast rows generated: 2820
+---------------+-------------+---------------+--------------------+---------------------------+
|station_id     |forecast_date|step_ahead_days|assumed_rainfall    |predicted_groundwater_level|
+---------------+-------------+---------------+--------------------+---------------------------+
|153145076380001|2023-05-26   |1              |2.8757142857142854  |3.7                        |
|154000075340001|2024-01-06   |1              |0.036333333333333336|5.8                        |
|154800075450001|2024-01-06   |1              |0.005172413793103449|10.53                      |
|155015074223001|2023-05-11   |1              |1.1078260869565215  |6.15                       |
|160324074195001|2023-05-11   |1              |1.0755555555555556  |1.08                       |
|161400074030001|2023-05-11   |1              |1.2967999999999997  |5.3                        |
|162430073594501|2023-05-11   |1              |0.8585714285714288  |5.3                   

In [ ]:
gold_df = (
    future_predictions_df
    .withColumn("model_name", F.lit(CHAMPION_MODEL_NAME))
    .withColumn("generated_at", F.current_timestamp())
)

if IS_DATABRICKS:
    gold_df.write.format("delta").mode("overwrite").save(GOLD_PATH)
    print("Gold Delta dataset saved on UC volume path.")
    print("Champion model:", CHAMPION_MODEL_NAME)
    print("Location:", GOLD_PATH)
else:
    spark.sql("CREATE DATABASE IF NOT EXISTS bhujal_mitra")

    gold_df.write.format("delta").mode("overwrite").save(GOLD_PATH)
    spark.sql(f"DROP TABLE IF EXISTS {GOLD_TABLE}")
    spark.sql(f"CREATE TABLE {GOLD_TABLE} USING DELTA LOCATION '{GOLD_PATH}'")

    print("Gold Delta table saved.")
    print("Champion model:", CHAMPION_MODEL_NAME)
    print("Table name:", GOLD_TABLE)
    print("Location:", GOLD_PATH)

Gold Delta table saved.
Champion model: naive_lag_1
Table name: bhujal_mitra.groundwater_prediction_gold
Location: /home/mridul-sharma/Desktop/bhujal-mitra/data/gold/groundwater_prediction_gold


In [ ]:
print("Holdout leaderboard (lower MAE/RMSE is better):")
spark.createDataFrame(metrics_pd.round(6)).show(truncate=False)

if 'rolling_summary_pd' in globals() and not rolling_summary_pd.empty:
    print("Rolling backtest summary (mean by model):")
    spark.createDataFrame(rolling_summary_pd.round(6)).show(truncate=False)
else:
    print("Rolling backtest summary not available.")

if IS_DATABRICKS:
    gold_loaded = spark.read.format("delta").load(GOLD_PATH)
else:
    gold_loaded = spark.read.format("delta").load(GOLD_PATH)

print(f"Gold rows: {gold_loaded.count()}")
print(f"Stations in Gold: {gold_loaded.select('station_id').distinct().count()}")

print("Per-station horizon validation:")
gold_loaded.groupBy("station_id").count().select(
    F.min("count").alias("min_forecast_days"),
    F.max("count").alias("max_forecast_days"),
    F.avg("count").alias("avg_forecast_days")
).show(truncate=False)

print("Prediction value range check:")
gold_loaded.select(
    F.min("predicted_groundwater_level").alias("min_predicted_level"),
    F.max("predicted_groundwater_level").alias("max_predicted_level"),
    F.avg("predicted_groundwater_level").alias("avg_predicted_level")
).show(truncate=False)

print("Null check in Gold:")
gold_loaded.select(
    F.sum(F.col("predicted_groundwater_level").isNull().cast("int")).alias("null_predictions"),
    F.sum(F.col("assumed_rainfall").isNull().cast("int")).alias("null_assumed_rainfall")
).show(truncate=False)

gold_loaded.orderBy("station_id", "forecast_date").show(20, truncate=False)

Holdout leaderboard (lower MAE/RMSE is better):
+------------------+--------+--------+--------+---------------------------+----------------------------+
|model_name        |rmse    |mae     |r2      |mae_improvement_vs_baseline|rmse_improvement_vs_baseline|
+------------------+--------+--------+--------+---------------------------+----------------------------+
|naive_lag_1       |4.581371|0.390525|0.946085|0.0                        |0.0                         |
|residual_gbt_v2   |4.615982|0.458446|0.945267|-0.067921                  |-0.034611                   |
|residual_linear_v2|3.874897|1.143418|0.961431|-0.752893                  |0.706474                    |
+------------------+--------+--------+--------+---------------------------+----------------------------+

Rolling backtest summary (mean by model):
+-----------+--------+--------+--------+---------------------------+----------------------------+
|model_name |mae     |rmse    |r2      |mae_improvement_vs_baseline|rmse_imp

In [ ]:
if IS_DATABRICKS:
    print("Skipping spark.stop() on Databricks managed runtime.")
else:
    spark.stop()
    print("Spark session stopped.")

Spark session stopped.
